# 01 - Análisis exploratorio de reservas hoteleras

Primera auditoría reproducible de `dataset_practica_final.csv`.

Objetivos: comprobar estructura y calidad, revisar duplicados, analizar `is_canceled`, generar gráficos y separar variables candidatas de fugas de información. Este notebook **no entrena todavía los modelos finales**.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette="deep")

DATA_PATH = Path("../data/raw/dataset_practica_final.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró {DATA_PATH.resolve()}")
DATA_PATH

## 1. Carga y comprobación estructural

In [ ]:
df = pd.read_csv(
    DATA_PATH,
    na_values=["NULL"],
    parse_dates=["reservation_status_date"],
)

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
display(df.head())

assert "is_canceled" in df.columns
assert set(df["is_canceled"].dropna().unique()).issubset({0, 1})
assert df.shape[1] == 32
print("Comprobaciones estructurales superadas.")

## 2. Tipos, cardinalidad y ausentes

In [ ]:
quality = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "ausentes": df.isna().sum(),
    "ausentes_pct": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True),
}).sort_values(["ausentes_pct", "valores_unicos"], ascending=[False, False])
display(quality)

In [ ]:
missing = quality.query("ausentes > 0").sort_values("ausentes_pct")
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=missing.reset_index(), x="ausentes_pct", y="index", ax=ax, color="#4472C4")
ax.set(title="Porcentaje de valores ausentes", xlabel="Ausentes (%)", ylabel="Variable")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=3)
plt.tight_layout()
plt.show()

## 3. Duplicados y controles de coherencia

In [ ]:
duplicates = int(df.duplicated().sum())
print(f"Duplicados exactos: {duplicates:,} ({duplicates / len(df):.2%})")
print("No se eliminan automáticamente: primero debe validarse su significado.")

checks = pd.Series({
    "reservas_sin_huespedes": ((df["adults"] + df["children"].fillna(0) + df["babies"]) == 0).sum(),
    "adr_negativo": (df["adr"] < 0).sum(),
    "adr_superior_1000": (df["adr"] > 1000).sum(),
    "estancias_cero_noches": ((df["stays_in_weekend_nights"] + df["stays_in_week_nights"]) == 0).sum(),
}, name="registros")
display(checks.to_frame())

## 4. Variable objetivo

In [ ]:
target_summary = (
    df["is_canceled"].value_counts().sort_index()
    .rename(index={0: "No cancelada", 1: "Cancelada"})
    .to_frame("reservas")
)
target_summary["porcentaje"] = (target_summary["reservas"] / len(df) * 100).round(2)
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="is_canceled", order=[0, 1], ax=ax, color="#4472C4")
ax.set(title="Distribución de cancelaciones", xlabel="Reserva cancelada", ylabel="Reservas")
ax.set_xticklabels(["No (0)", "Sí (1)"])
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.tight_layout()
plt.show()

## 5. Distribuciones numéricas

In [ ]:
numeric_focus = [
    "lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "adr", "previous_cancellations",
    "total_of_special_requests",
]
display(df[numeric_focus].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for variable, ax in zip(
    ["lead_time", "adr", "stays_in_week_nights", "total_of_special_requests"],
    axes.flat,
):
    upper = df[variable].quantile(0.99)
    sns.histplot(df.loc[df[variable] <= upper, variable], bins=40, ax=ax, color="#4472C4")
    ax.set_title(f"{variable} (hasta percentil 99)")
plt.tight_layout()
plt.show()

## 6. Cancelación por categorías de negocio

In [ ]:
def cancellation_rate_plot(data, variable, max_categories=15):
    summary = (
        data.groupby(variable, dropna=False)["is_canceled"]
        .agg(reservas="size", tasa_cancelacion="mean")
        .sort_values("reservas", ascending=False)
        .head(max_categories)
        .sort_values("tasa_cancelacion")
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(9, max(3.5, len(summary) * 0.38)))
    sns.barplot(data=summary, x="tasa_cancelacion", y=variable, ax=ax, color="#ED7D31")
    ax.set(title=f"Tasa de cancelación por {variable}", xlabel="Tasa", ylabel=variable)
    ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    plt.tight_layout()
    plt.show()
    return summary

for variable in ["hotel", "deposit_type", "market_segment", "customer_type"]:
    display(cancellation_rate_plot(df, variable))

## 7. Estacionalidad

In [ ]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
monthly = (
    df.groupby("arrival_date_month")["is_canceled"]
    .agg(reservas="size", tasa_cancelacion="mean")
    .reindex(month_order)
    .reset_index()
)
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly, x="arrival_date_month", y="tasa_cancelacion", marker="o", ax=ax, color="#C00000")
ax.set(title="Tasa de cancelación por mes", xlabel="Mes", ylabel="Tasa")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
display(monthly)

## 8. Grupos de variables y fuga de información

In [ ]:
target = "is_canceled"
leakage_features = ["reservation_status", "reservation_status_date"]
time_dependent_features = ["assigned_room_type", "booking_changes", "days_in_waiting_list"]
high_missing_features = ["company"]
candidate_features = [
    column for column in df.columns
    if column not in [target, *leakage_features, *time_dependent_features, *high_missing_features]
]

feature_groups = pd.DataFrame({
    "grupo": ["Objetivo", "Excluir por fuga", "Validar temporalmente", "Excluir inicialmente por ausentes", "Candidatas"],
    "variables": [target, ", ".join(leakage_features), ", ".join(time_dependent_features), ", ".join(high_missing_features), ", ".join(candidate_features)],
})
display(feature_groups)
print(f"Candidatas iniciales: {len(candidate_features)}")

## 9. Asociación numérica inicial

In [ ]:
numeric_candidates = [column for column in candidate_features if pd.api.types.is_numeric_dtype(df[column])]
correlations = (
    df[numeric_candidates + [target]].corr(numeric_only=True)[target]
    .drop(target).sort_values(key=lambda values: values.abs(), ascending=False)
    .rename("correlacion_con_is_canceled").to_frame()
)
display(correlations)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=correlations.reset_index(), x="correlacion_con_is_canceled", y="index", ax=ax, color="#70AD47")
ax.set(title="Correlación lineal inicial", xlabel="Correlación", ylabel="Variable")
plt.tight_layout()
plt.show()

## 10. Decisiones posteriores al EDA

El equipo deberá justificar el tratamiento de duplicados y ausentes, el momento de predicción, las variables condicionadas temporalmente, las categorías de alta cardinalidad, los valores extremos y el tipo de partición de entrenamiento y prueba.

Las decisiones se validarán más adelante mediante pipelines y métricas comunes para todos los modelos.